In [1]:
# ==========================================================
# PIPELINE COMPLETO - Wine Quality Classification
# Dataset: https://www.kaggle.com/datasets/yasserh/wine-quality-dataset
#
# Modelos:

In [2]:
# - Logistic Regression
# - Ridge Classifier
# - Linear Discriminant Analysis
# - Decision Tree
# - Random Forest
# - Extra Trees
# - AdaBoost
# - Gradient Boosting
# - XGBoost
# - LightGBM
# - CatBoost
# - Naive Bayes
# - K Neighbors
# - Support Vector Machine
# - MLP (sklearn)
# - PyTorch MLP
# - PyTorch CNN
# - PyTorch RNN (LSTM)
#
# GPU: automático (cuda se existir)
# Saída: wine_classico/
# ==========================================================

In [3]:
import os
import json
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

import kagglehub
from ydata_profiling import ProfileReport
import joblib

# -------------------------
# Boosting avançado
# -------------------------
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None

# -------------------------
# PyTorch
# -------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo PyTorch:", DEVICE)

# -------------------------
# Diretórios
# -------------------------
BASE_DIR = os.path.abspath("wine_classico")
DIR_REPORTS = os.path.join(BASE_DIR, "reports")
DIR_FIGURES = os.path.join(BASE_DIR, "figures")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DIR_REPORTS, exist_ok=True)
os.makedirs(DIR_FIGURES, exist_ok=True)

Dispositivo PyTorch: cuda


In [4]:
# -------------------------
# Auxiliares
# -------------------------
def load_dataset():
    path = kagglehub.dataset_download("yasserh/wine-quality-dataset")
    csv = [c for c in glob.glob(os.path.join(path, "*.csv")) if "WineQT" in c][0]
    return pd.read_csv(csv).drop(columns=["Id"])


def feature_engineering(df):
    eps = 1e-9
    df = df.copy()
    df["total_acidity"] = df["fixed acidity"] + df["volatile acidity"] + df["citric acid"]
    df["alcohol_sugar_ratio"] = df["alcohol"] / (df["residual sugar"] + eps)
    df["so2_ratio"] = df["free sulfur dioxide"] / (df["total sulfur dioxide"] + eps)
    df["density_alcohol"] = df["density"] * df["alcohol"]
    return df


def make_preprocessor(columns):
    """Cria um preprocessor NOVO (para evitar efeitos colaterais entre modelos)."""
    return ColumnTransformer(
        [("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), columns)]
    )


# -------------------------
# PyTorch Models
# -------------------------
class TorchMLP(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.net(x)


class TorchCNN(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.conv = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * in_dim, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)


class TorchRNN(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, n_classes)

    def forward(self, x):
        x = x.unsqueeze(-1)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])


def train_torch(model, loader, epochs=20):
    model.to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()


# -------------------------
# MAIN
# -------------------------
def main():
    print("[1] Carregando dados...")
    df = load_dataset()

    print("[2] EDA...")
    ProfileReport(df, title="Wine Quality EDA", explorative=True)\
        .to_file(f"{DIR_REPORTS}/eda_wine.html")

    df = feature_engineering(df)

    # Encoding do target (classes 0..n-1)
    le = LabelEncoder()
    y = le.fit_transform(df["quality"].astype(int))
    X = df.drop(columns=["quality"])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    # ------------------------------------------
    # Modelos sklearn (cada pipeline com seu prep)
    # ------------------------------------------
    models = {
        "LogisticRegression": LogisticRegression(max_iter=5000, multi_class="multinomial"),
        "RidgeClassifier": RidgeClassifier(),
        "LDA": LinearDiscriminantAnalysis(),
        "DecisionTree": DecisionTreeClassifier(),
        "RandomForest": RandomForestClassifier(n_estimators=400),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=400),
        "AdaBoost": AdaBoostClassifier(),
        "GradientBoosting": GradientBoostingClassifier(),
        "NaiveBayes": GaussianNB(),
        "KNN": KNeighborsClassifier(15),
        "SVM": SVC(probability=True),
        "MLP_sklearn": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=2000)
    }

    if XGBClassifier:
        models["XGBoost"] = XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_estimators=300
        )

    if LGBMClassifier:
        models["LightGBM"] = LGBMClassifier(objective="multiclass")

    if CatBoostClassifier:
        models["CatBoost"] = CatBoostClassifier(loss_function="MultiClass", verbose=False)

    results = {}
    fitted = {}

    print("[3] Modelos clássicos (sklearn)...")
    for name, model in models.items():
        prep = make_preprocessor(X.columns)  # ✅ preprocessor novo
        pipe = Pipeline([("prep", prep), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        f1 = f1_score(y_test, pred, average="macro")
        results[name] = f1
        fitted[name] = pipe
        print(f"{name:20s} | F1-macro: {f1:.4f}")

    # ------------------------------------------
    # Modelos PyTorch (com preprocessor próprio)
    # ------------------------------------------
    print("[4] Modelos PyTorch...")
    preprocessor_torch = make_preprocessor(X.columns)  # ✅ separado do sklearn
    X_train_t = torch.tensor(preprocessor_torch.fit_transform(X_train), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=64, shuffle=True
    )

    n_classes = len(np.unique(y))
    torch_models = {
        "Torch_MLP": TorchMLP(X_train_t.shape[1], n_classes),
        "Torch_CNN": TorchCNN(X_train_t.shape[1], n_classes),
        "Torch_RNN": TorchRNN(X_train_t.shape[1], n_classes)
    }

    for name, mdl in torch_models.items():
        train_torch(mdl, train_loader)
        with torch.no_grad():
            X_test_t = torch.tensor(
                preprocessor_torch.transform(X_test),
                dtype=torch.float32
            ).to(DEVICE)
            pred = mdl(X_test_t).argmax(1).cpu().numpy()
            f1 = f1_score(y_test, pred, average="macro")
            results[name] = f1
            fitted[name] = mdl
            print(f"{name:20s} | F1-macro: {f1:.4f}")

    best_model_name = max(results, key=results.get)
    print("\n✅ MELHOR MODELO:", best_model_name)

    # ==========================================================
    # ✅ EXPORTAÇÃO PADRONIZADA PARA STREAMLIT ÚNICO
    # - Sempre gera meta.json com features
    # - Se melhor for sklearn: salva model_streamlit.joblib (Pipeline)
    # - Se melhor for torch: salva bundle torch_* (weights + prep + encoder + info)
    # ==========================================================
    meta = {
        "target": "quality",
        "features": X.columns.tolist(),
        "classes": list(range(len(le.classes_))),
        "original_classes": le.classes_.tolist(),
        "best_model": best_model_name,
        "scores": results
    }

    if not best_model_name.startswith("Torch_"):
        artifact_name = "model_streamlit.joblib"
        joblib.dump(fitted[best_model_name], os.path.join(BASE_DIR, artifact_name))
        meta.update({
            "model_kind": "sklearn",
            "artifact_path": artifact_name
        })
        print("✅ Exportado para Streamlit: model_streamlit.joblib")

    else:
        # Salva preprocessor do torch
        joblib.dump(preprocessor_torch, os.path.join(BASE_DIR, "torch_preprocessor.joblib"))
        # Salva label encoder
        joblib.dump(le, os.path.join(BASE_DIR, "label_encoder.joblib"))
        # Salva pesos da rede
        torch.save(fitted[best_model_name].state_dict(), os.path.join(BASE_DIR, "torch_model.pt"))

        torch_info = {
            "arch": best_model_name,                 # Torch_MLP / Torch_CNN / Torch_RNN
            "input_dim": int(X_train_t.shape[1]),
            "n_classes": int(n_classes)
        }
        with open(os.path.join(BASE_DIR, "torch_info.json"), "w", encoding="utf-8") as f:
            json.dump(torch_info, f, indent=2, ensure_ascii=False)

        meta.update({
            "model_kind": "torch",
            "artifact_path": {
                "state_dict": "torch_model.pt",
                "preprocessor": "torch_preprocessor.joblib",
                "label_encoder": "label_encoder.joblib",
                "torch_info": "torch_info.json"
            }
        })
        print("✅ Exportado para Streamlit: bundle torch_*")

    with open(os.path.join(BASE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print("✅ Artefatos salvos em wine_classico/")
    print(" - wine_classico/meta.json")
    print(" - wine_classico/model_streamlit.joblib  (se sklearn)")
    print(" - wine_classico/torch_*                (se torch)")


if __name__ == "__main__":
    main()

[1] Carregando dados...
[2] EDA...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

[3] Modelos clássicos (sklearn)...
LogisticRegression   | F1-macro: 0.2890
RidgeClassifier      | F1-macro: 0.2397
LDA                  | F1-macro: 0.3297
DecisionTree         | F1-macro: 0.3337
RandomForest         | F1-macro: 0.3443
ExtraTrees           | F1-macro: 0.3414
AdaBoost             | F1-macro: 0.2367
GradientBoosting     | F1-macro: 0.3619
NaiveBayes           | F1-macro: 0.3026
KNN                  | F1-macro: 0.2827
SVM                  | F1-macro: 0.2955
MLP_sklearn          | F1-macro: 0.3168
XGBoost              | F1-macro: 0.3282
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000771 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1964
[LightGBM] [Info] Number of data points in the train set: 914, number of used features: 15
[LightGBM] [Info] Start training from score -5.208393
[LightGBM] [Info] Start training from sc

  File "C:\Users\calab\.venv310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\calab\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\calab\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\calab\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [3]:
import os
for item in os.listdir():
    caminho = os.path.join(os.getcwd(), item)
print(caminho)

D:\TreinaRecife\Python do Zero até a Análise de Dados\aprendizado\Códigos\wine_tpot.ipynb


In [2]:
caminho

'D:\\TreinaRecife\\Python do Zero até a Análise de Dados\\aprendizado\\Códigos\\wine_tpot.ipynb'